# Qwen3.5-4B QLoRA: vLLM Dual-GPU

This independent training notebook reuses the existing IR4 input protocol and embeds the complete five-fold cache. Create the package locally, copy its printed `manifest_sha256` into `EXPECTED_MANIFEST_SHA256` in the first code cell, and attach that exact `qwen35_4b.zip` as a private Kaggle input. This authenticates the manifest before any project code is copied or installed.

Training and evaluation-under-training stay on Transformers, because vLLM is inference-only. Adapter reload checks, dev/confirm evaluation, and test inference run on **vLLM 0.24.0 with tensor parallelism across both T4 GPUs**. Both engines share the same data contract and constrained answer space, and the run contract records which engine produced each result.

The package contains no model weights. Use a Kaggle 2x T4 session; the notebook verifies that both GPUs are visible before vLLM starts.


In [ ]:
from pathlib import Path, PurePosixPath
from collections import deque
import hashlib, json, os, re, shutil, subprocess, sys, tempfile, zipfile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
BUNDLE_INPUT = None
EXPECTED_MANIFEST_SHA256 = None  # paste manifest_sha256 from the trusted package command  # ZIP，或含 qwen35_training_bundle_manifest.json 的目录
WEIGHTS_INPUT = None  # 可选：含 cuhkx_qwen35_weights.json 的完整 Qwen3.5 权重目录
EXPERIMENT = "qwen35_vllm_full_v1"
TRAIN_GPU = 0          # 训练固定单卡，避免与 vLLM 的 TP 进程争抢显存
TENSOR_PARALLEL = 2    # vLLM 张量并行度：2×T4
GPU_MEMORY_UTILIZATION = 0.80
RUN_CONFIRMATION = False
RUN_TEST = False
PACKAGE_ID = "cuhkx-qwen35-4b-vllm-full-v1"
MARKER = "qwen35_training_bundle_manifest.json"


## 1. 验证独立训练包并建立隔离工作目录


In [ ]:
import hmac, stat

MAX_ARCHIVE_BYTES = 512 * 1024**2
MAX_ARCHIVE_FILES = 20_000
MAX_EXPANDED_BYTES = 512 * 1024**2
MAX_MEMBER_BYTES = 16 * 1024**2
MAX_COMPRESSION_RATIO = 50.0
FREE_SPACE_RESERVE = 256 * 1024**2

if not isinstance(EXPECTED_MANIFEST_SHA256, str) or re.fullmatch(r"[0-9a-f]{64}", EXPECTED_MANIFEST_SHA256) is None:
    raise RuntimeError("set EXPECTED_MANIFEST_SHA256 from the trusted local package command")


def _safe_bundle_name(name):
    path = PurePosixPath(name)
    if (not name or path.is_absolute() or ".." in path.parts or "\\" in name or ":" in name
            or path.as_posix() != name or any(ord(character) < 32 for character in name)):
        raise RuntimeError("unsafe package path: " + repr(name))
    return path


def _open_bounded_archive(bundle):
    if not bundle.is_file():
        return None, None, 0
    if bundle.stat().st_size > MAX_ARCHIVE_BYTES:
        raise RuntimeError("package exceeds compressed-size budget")
    archive = zipfile.ZipFile(bundle)
    infos = archive.infolist()
    if not infos or len(infos) > MAX_ARCHIVE_FILES:
        raise RuntimeError("package entry count is outside the allowed budget")
    index, folded, expanded = {}, set(), 0
    for info in infos:
        _safe_bundle_name(info.filename)
        folded_name = info.filename.casefold()
        if info.filename in index or folded_name in folded:
            raise RuntimeError("package contains duplicate or case-colliding paths")
        mode = (info.external_attr >> 16) & 0xFFFF
        if (info.is_dir() or info.flag_bits & 1 or stat.S_IFMT(mode) not in (0, stat.S_IFREG)
                or info.compress_type not in (zipfile.ZIP_STORED, zipfile.ZIP_DEFLATED)):
            raise RuntimeError("package contains an unsupported entry")
        if info.file_size < 0 or info.file_size > MAX_MEMBER_BYTES:
            raise RuntimeError("package member exceeds size budget")
        if info.file_size and (not info.compress_size
                or info.file_size / info.compress_size > MAX_COMPRESSION_RATIO):
            raise RuntimeError("package member exceeds compression-ratio budget")
        expanded += info.file_size
        if expanded > MAX_EXPANDED_BYTES:
            raise RuntimeError("package exceeds expanded-size budget")
        index[info.filename] = info
        folded.add(folded_name)
    return archive, index, expanded


def _directory_member(bundle, name):
    source = bundle / name
    if source.is_symlink():
        raise RuntimeError("package symlinks are unsupported: " + name)
    path = source.resolve()
    if not path.is_relative_to(bundle.resolve()) or not path.is_file():
        raise RuntimeError("unsafe package file: " + name)
    if path.stat().st_size > MAX_MEMBER_BYTES:
        raise RuntimeError("package member exceeds size budget")
    return path


def _member_bytes(bundle, archive, index, name):
    if archive:
        info = index.get(name)
        if info is None:
            raise RuntimeError("package member is missing: " + name)
        with archive.open(info) as handle:
            content = handle.read(MAX_MEMBER_BYTES + 1)
        if len(content) != info.file_size or len(content) > MAX_MEMBER_BYTES:
            raise RuntimeError("package member size changed while reading")
        return content
    return _directory_member(bundle, name).read_bytes()


def _trusted_manifest(bundle, archive, index, marker):
    raw = _member_bytes(bundle, archive, index, marker)
    digest = hashlib.sha256(raw).hexdigest()
    if not hmac.compare_digest(digest, EXPECTED_MANIFEST_SHA256):
        raise RuntimeError("package manifest is not the trusted release")
    return raw, json.loads(raw)


def _validated_bundle_entries(bundle, archive, index, marker, manifest, prefix):
    entries = manifest.get("files")
    if not isinstance(entries, list) or len(entries) > MAX_ARCHIVE_FILES - 1:
        raise RuntimeError("invalid package file list")
    expected, folded, total = {}, set(), 0
    for entry in entries:
        if not isinstance(entry, dict) or set(entry) != {"path", "bytes", "sha256"}:
            raise RuntimeError("invalid package entry")
        name, size, digest = entry["path"], entry["bytes"], entry["sha256"]
        path = _safe_bundle_name(name)
        if (not name.startswith(prefix) or not isinstance(size, int) or isinstance(size, bool)
                or size < 0 or size > MAX_MEMBER_BYTES or not isinstance(digest, str)
                or re.fullmatch(r"[0-9a-f]{64}", digest) is None):
            raise RuntimeError("invalid package path, size, or digest")
        if name in expected or name.casefold() in folded:
            raise RuntimeError("package manifest contains duplicate paths")
        if archive:
            if name not in index or index[name].file_size != size:
                raise RuntimeError("package manifest differs from ZIP metadata")
        else:
            if _directory_member(bundle, name).stat().st_size != size:
                raise RuntimeError("package manifest differs from directory metadata")
        expected[name] = entry
        folded.add(name.casefold())
        total += size
        if total > MAX_EXPANDED_BYTES:
            raise RuntimeError("package manifest exceeds expanded-size budget")
    actual = set(index) if archive else {marker, *expected}
    if actual != set(expected) | {marker}:
        raise RuntimeError("package file set differs from manifest")
    disk_root = WORK
    while not disk_root.exists():
        disk_root = disk_root.parent
    if shutil.disk_usage(disk_root).free < total + FREE_SPACE_RESERVE:
        raise RuntimeError("insufficient free space for bounded extraction")
    return list(expected.values())


def _verify_entry(bundle, archive, index, entry):
    digest, size = hashlib.sha256(), 0
    source = archive.open(index[entry["path"]]) if archive else _directory_member(bundle, entry["path"]).open("rb")
    with source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            size += len(block)
            if size > entry["bytes"] or size > MAX_MEMBER_BYTES:
                raise RuntimeError("package member exceeded manifest size")
            digest.update(block)
    if size != entry["bytes"] or digest.hexdigest() != entry["sha256"]:
        raise RuntimeError("package content hash mismatch: " + entry["path"])


def _copy_entry(bundle, archive, index, entry, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + ".partial")
    if partial.exists():
        partial.unlink()
    digest, size = hashlib.sha256(), 0
    source = archive.open(index[entry["path"]]) if archive else _directory_member(bundle, entry["path"]).open("rb")
    try:
        with source, partial.open("xb") as output:
            for block in iter(lambda: source.read(1024 * 1024), b""):
                size += len(block)
                if size > entry["bytes"] or size > MAX_MEMBER_BYTES:
                    raise RuntimeError("package member exceeded manifest size")
                digest.update(block)
                output.write(block)
        if size != entry["bytes"] or digest.hexdigest() != entry["sha256"]:
            raise RuntimeError("package content changed while copying")
        os.replace(partial, target)
    finally:
        if partial.exists():
            partial.unlink()

PACKAGE_ID = 'cuhkx-qwen35-4b-vllm-full-v1'
MARKER = 'qwen35_training_bundle_manifest.json'
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", EXPERIMENT):
    raise RuntimeError("invalid experiment name")
if BUNDLE_INPUT is None:
    candidates = []
    for candidate_marker in INPUT.rglob(MARKER):
        try:
            if candidate_marker.stat().st_size <= MAX_MEMBER_BYTES:
                value = json.loads(candidate_marker.read_text(encoding="utf-8"))
                if value.get('package_id') == PACKAGE_ID:
                    candidates.append(candidate_marker.parent)
        except (OSError, ValueError):
            pass
    if not candidates:
        candidates = list(INPUT.rglob('qwen35_4b.zip'))
    if len(candidates) != 1:
        raise RuntimeError(f"found {len(candidates)} matching packages; set BUNDLE_INPUT")
    BUNDLE_INPUT = candidates[0]
BUNDLE_INPUT = Path(BUNDLE_INPUT).resolve()
archive, archive_index, expanded_bytes = _open_bounded_archive(BUNDLE_INPUT)
try:
    raw_manifest, manifest = _trusted_manifest(
        BUNDLE_INPUT, archive, archive_index, MARKER)
    if manifest.get("schema_version") != 1 or manifest.get('package_id') != PACKAGE_ID:
        raise RuntimeError("wrong package identity")
    if manifest.get("training_cache_mode") != "embedded_complete":
        raise RuntimeError("training package does not contain the complete cache")
    if manifest.get("inference_engine") != "vllm_0.24.0_tensor_parallel":
        raise RuntimeError("training package was not built for the vLLM dual-GPU lane")
    entries = _validated_bundle_entries(
        BUNDLE_INPUT, archive, archive_index, MARKER, manifest, 'qwen35_training_repo/')
    for entry in entries:
        _verify_entry(BUNDLE_INPUT, archive, archive_index, entry)
    MANIFEST_SHA256 = hashlib.sha256(raw_manifest).hexdigest()
    WORK_ROOT = WORK.resolve()
    RUNTIME = WORK_ROOT / ('qwen35_qlora_' + MANIFEST_SHA256[:12]) / EXPERIMENT
    if RUNTIME.is_symlink():
        raise RuntimeError("runtime root must not be a symlink")
    RUNTIME_ROOT = RUNTIME.resolve()
    if not RUNTIME_ROOT.is_relative_to(WORK_ROOT):
        raise RuntimeError("runtime root escapes working directory")
    for entry in entries:
        candidate = RUNTIME_ROOT / entry["path"]
        target = candidate.resolve()
        if candidate.is_symlink() or not target.is_relative_to(RUNTIME_ROOT):
            raise RuntimeError("runtime path escapes package root")
        if target.exists():
            if target.is_symlink() or not target.is_file():
                raise RuntimeError("runtime contains an unsafe existing path")
            digest = hashlib.sha256(target.read_bytes()).hexdigest()
            if target.stat().st_size != entry["bytes"] or digest != entry["sha256"]:
                raise RuntimeError("runtime copy was modified; use a new experiment/runtime")
    for entry in entries:
        target = (RUNTIME_ROOT / entry["path"]).resolve()
        if not target.exists():
            _copy_entry(BUNDLE_INPUT, archive, archive_index, entry, target)
    REPO = RUNTIME_ROOT / 'qwen35_training_repo'
    (RUNTIME_ROOT / MARKER).write_bytes(raw_manifest)
finally:
    if archive:
        archive.close()
print("Verified repository:", REPO)
print("Trusted manifest SHA256:", MANIFEST_SHA256)
print("Package:", PACKAGE_ID)


## 2. 安装独立 Python 3.11 环境并检查完整五折数据


In [ ]:
VENV = RUNTIME / "train_env"
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    with tempfile.TemporaryDirectory(prefix="cuhkx_uv_bootstrap_", dir=WORK) as bootstrap_dir:
        BOOT = Path(bootstrap_dir)
        subprocess.run([sys.executable, "-m", "pip", "install", "--target", str(BOOT),
                        "--no-deps", "--require-hashes", "--only-binary=:all:",
                        "-r", str(REPO / "requirements/bootstrap.lock.txt")], check=True)
        subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.11",
                        "--seed", str(VENV)], env={**os.environ, "PYTHONPATH": str(BOOT)}, check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--require-hashes",
                "--only-binary=:all:", "-r", str(REPO / "requirements/cpu.lock.txt")], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-deps",
                "--no-build-isolation", "-e", str(REPO)], check=True)

def command(*args):
    return [str(PYTHON), "-m", "cuhkx.cli", *args, "--project-root", str(REPO)]

CLOUD_ENV = {**os.environ, "PYTHONPATH": str(REPO / "src"),
             "PYTHONDONTWRITEBYTECODE": "1", "PYTHONUNBUFFERED": "1",
             "CUHKX_TRACEBACK": "1"}

def cloud(*args):
    tail = deque(maxlen=120)
    with subprocess.Popen(command(*args), cwd=REPO, env=CLOUD_ENV,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            tail.append(line)
        code = process.wait()
    if code:
        raise RuntimeError(f"{args[0]} exited with code {code}; last output:\n" + "".join(tail))

TRAINING_CONFIG = REPO / "configs/training_qwen35.yaml"
data_state = json.loads(subprocess.check_output(command(
    "training-check", "--profile", "qwen35", "--training-config", str(TRAINING_CONFIG)),
    cwd=REPO, env=CLOUD_ENV, text=True))
if data_state["status"] != "PASS":
    raise RuntimeError("embedded five-fold caches are incomplete")
print(json.dumps(data_state["coverage"], indent=2))


## 3. 安装 Qwen3.5 后训练依赖（含 vLLM 0.24.0）并固定双卡环境


In [ ]:
subprocess.run([str(PYTHON), "-m", "pip", "install", "--require-hashes",
                "--only-binary=:all:", "--index-url", "https://pypi.org/simple",
                "--extra-index-url", "https://download.pytorch.org/whl/cu126",
                "-r", str(REPO / "requirements/train_qwen35.lock.txt")], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
compatibility_probe = (
    "import json, peft, transformers, vllm, torch; "
    "from transformers import AutoModelForMultimodalLM, Trainer; "
    "print(json.dumps({'transformers': transformers.__version__, 'peft': peft.__version__, "
    "'vllm': vllm.__version__, 'torch': torch.__version__, "
    "'multimodal_class': AutoModelForMultimodalLM.__name__}))"
)
print(subprocess.check_output([str(PYTHON), "-c", compatibility_probe], text=True))
probe = ("import json,sys,torch; assert sys.version_info[:2]==(3,11); "
         "assert torch.cuda.is_available(), 'a cloud CUDA GPU is required'; "
         "count=torch.cuda.device_count(); "
         "assert count>=TENSOR_PARALLEL_COUNT, f'vLLM TP={TENSOR_PARALLEL_COUNT} needs that many GPUs'; "
         "print(json.dumps({'python':sys.version,'torch':torch.__version__,"
         "'cuda':torch.version.cuda,'device_count':count,"
         "'devices':[torch.cuda.get_device_name(i) for i in range(count)]}))")
environment = subprocess.check_output(
    [str(PYTHON), "-c", f"TENSOR_PARALLEL_COUNT={TENSOR_PARALLEL};{probe}"], text=True)
(RUNTIME / "environment.json").write_text(environment, encoding="utf-8")
(RUNTIME / "environment.freeze.txt").write_text(
    subprocess.check_output([str(PYTHON), "-m", "pip", "freeze", "--all"], text=True),
    encoding="utf-8")
print(environment)


## 4. 解析并固定 Qwen3.5 revision，下载或复用已验证权重


In [ ]:
profile_path = REPO / "configs/qwen35_4b.yaml"
profile = json.loads(json.dumps(__import__("yaml").safe_load(profile_path.read_text(encoding="utf-8"))))
if WEIGHTS_INPUT is None:
    candidates = []
    for receipt_path in INPUT.rglob("cuhkx_qwen35_weights.json"):
        try:
            value = json.loads(receipt_path.read_text(encoding="utf-8"))
            if value.get("model_id") == "Qwen/Qwen3.5-4B":
                candidates.append(receipt_path.parent)
        except (OSError, ValueError):
            pass
    if len(candidates) > 1:
        raise RuntimeError("found multiple Qwen3.5 weight directories; set WEIGHTS_INPUT")
    WEIGHTS = candidates[0] if candidates else Path("/tmp") / "qwen35_4b_weights"
else:
    WEIGHTS = Path(WEIGHTS_INPUT)
if not (WEIGHTS / "cuhkx_qwen35_weights.json").exists():
    from huggingface_hub import HfApi
    revision = HfApi().model_info("Qwen/Qwen3.5-4B").sha
    if not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise RuntimeError("Hugging Face did not return an immutable revision")
else:
    revision = json.loads((WEIGHTS / "cuhkx_qwen35_weights.json").read_text(encoding="utf-8"))["revision"]
profile["model"]["revision"] = revision
profile_path.write_text(__import__("yaml").safe_dump(profile, sort_keys=False), encoding="utf-8")
cloud("fetch-qwen35-weights", "--weights-dir", str(WEIGHTS), "--revision", revision)
cloud("check", "--profile", "qwen35", "--dataset", "test")
cloud("check", "--profile", "qwen35", "--dataset", "pilot")


## 5. 训练与训练中评估走 Transformers（vLLM 不能训练）


In [ ]:
cloud("train", "--profile", "qwen35", "--training-config", str(TRAINING_CONFIG),
      "--weights-dir", str(WEIGHTS), "--run-id", "qwen35_pt_smoke", "--smoke-steps", "4",
      "--gpu", str(TRAIN_GPU), "--resume")
SMOKE_ADAPTER = REPO / "artifacts/training/qwen35_pt_smoke/adapter"
cloud("train", "--profile", "qwen35", "--training-config", str(TRAINING_CONFIG),
      "--weights-dir", str(WEIGHTS), "--run-id", "qwen35_pt_sft", "--gpu", str(TRAIN_GPU), "--resume")
ADAPTER = REPO / "artifacts/training/qwen35_pt_sft/adapter"


## 6. vLLM 双卡短跑：验证 TP=2 引擎、答案约束与 adapter 重载


In [ ]:
cloud("predict", "--profile", "qwen35", "--backend", "vllm",
      "--tensor-parallel-size", str(TENSOR_PARALLEL),
      "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
      "--dataset", "pilot", "--limit", "16",
      "--weights-dir", str(WEIGHTS), "--adapter-dir", str(SMOKE_ADAPTER),
      "--run-id", "qwen35_vllm_smoke_reload", "--resume")
smoke = json.loads((REPO / "outputs/qwen35_vllm_smoke_reload/run_summary.json").read_text())
backend_metadata = smoke["backend"]
if backend_metadata.get("backend") != "qwen35_4b_vllm":
    raise RuntimeError("smoke run did not use the vLLM engine")
if backend_metadata.get("tensor_parallel_size") != TENSOR_PARALLEL:
    raise RuntimeError(f"vLLM ran with TP={backend_metadata.get('tensor_parallel_size')}, expected {TENSOR_PARALLEL}")
print(json.dumps(backend_metadata, indent=2))


## 7. vLLM dev 基座/候选对照与 confirm 门禁


In [ ]:
VLLM = ["--backend", "vllm", "--tensor-parallel-size", str(TENSOR_PARALLEL),
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION)]
cloud("evaluate-training", "--profile", "qwen35", "--training-config", str(TRAINING_CONFIG),
      *VLLM, "--split", "dev", "--weights-dir", str(WEIGHTS),
      "--run-id", "qwen35_pt_base_dev", "--resume")
cloud("evaluate-training", "--profile", "qwen35", "--training-config", str(TRAINING_CONFIG),
      *VLLM, "--split", "dev", "--weights-dir", str(WEIGHTS), "--adapter-dir", str(ADAPTER),
      "--run-id", "qwen35_pt_adapter_dev", "--resume")
def accuracy(run_id):
    return json.loads((REPO / "outputs" / run_id / "metrics.json").read_text())["metrics"]["overall_accuracy"]
print("vLLM dev baseline:", accuracy("qwen35_pt_base_dev"), "adapter:", accuracy("qwen35_pt_adapter_dev"))

CONFIRMED = False
if RUN_CONFIRMATION:
    cloud("verify-run", "--profile", "qwen35", "--training-config", str(TRAINING_CONFIG),
          "--run-id", "qwen35_pt_base_dev")
    cloud("verify-run", "--profile", "qwen35", "--training-config", str(TRAINING_CONFIG),
          "--run-id", "qwen35_pt_adapter_dev")
    if accuracy("qwen35_pt_adapter_dev") <= accuracy("qwen35_pt_base_dev"):
        raise RuntimeError("adapter has no dev improvement; formal test is gated")
    cloud("evaluate-training", "--profile", "qwen35", "--training-config", str(TRAINING_CONFIG),
          *VLLM, "--split", "confirm", "--weights-dir", str(WEIGHTS),
          "--run-id", "qwen35_pt_base_confirm", "--resume")
    cloud("evaluate-training", "--profile", "qwen35", "--training-config", str(TRAINING_CONFIG),
          *VLLM, "--split", "confirm", "--weights-dir", str(WEIGHTS), "--adapter-dir", str(ADAPTER),
          "--run-id", "qwen35_pt_adapter_confirm", "--resume")
    CONFIRMED = accuracy("qwen35_pt_adapter_confirm") > accuracy("qwen35_pt_base_confirm")
    print("confirm improved:", CONFIRMED)


## 8. 只有 confirm 提升后才用 vLLM 导出 test submission


In [ ]:
if RUN_TEST:
    if not RUN_CONFIRMATION or not CONFIRMED:
        raise RuntimeError("confirm gate is not satisfied")
    cloud("verify-run", "--profile", "qwen35", "--training-config", str(TRAINING_CONFIG),
          "--run-id", "qwen35_pt_adapter_confirm")
    cloud("predict", "--profile", "qwen35", *VLLM, "--dataset", "test", "--weights-dir", str(WEIGHTS),
          "--adapter-dir", str(ADAPTER), "--run-id", "qwen35_pt_test", "--resume")
    cloud("submit", "--profile", "qwen35", "--run-id", "qwen35_pt_test")
    print("submission:", REPO / "outputs/qwen35_pt_test/submission.csv")
print("adapter/checkpoints:", REPO / "artifacts/training")
print("run evidence:", REPO / "outputs")
print("runtime/environment backup:", RUNTIME)
